# A Fool Fraud — Traditional Prediction Models
**Credit Card Fraud Detection using Machine Learning**

This notebook builds and evaluates traditional ML classifiers on the credit card fraud dataset.
It serves as the **baseline** to be compared against data-augmented models.

### Classifiers evaluated:
- Logistic Regression
- Random Forest
- Support Vector Machine (SVM)
- Gradient Boosting Machine (GBM)

### Evaluation metrics (imbalance-aware):
- Precision, Recall, F1-Score
- ROC-AUC
- Confusion Matrix

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn - Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Sklearn - Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Sklearn - Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('All libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
# Load dataset — place creditcard.csv in the same directory as this notebook
# Alternatively, download it:
# !wget https://storage.googleapis.com/qwasar-public/track-ds/creditcard.csv

df = pd.read_csv('creditcard.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['Class'].value_counts())
print(f'\nFraud rate: {df["Class"].mean() * 100:.4f}%')

df.head()

## 3. Preprocessing

In [ ]:
# Check for nulls
print('Missing values per column:')
print(df.isnull().sum().sum(), 'total missing values')

# The 'Amount' and 'Time' features are not PCA-transformed — scale them
# V1-V28 are already PCA-transformed (mean≈0, unit variance)

scaler = StandardScaler()

df['scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_Time']   = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop originals
df_clean = df.drop(['Amount', 'Time'], axis=1)

print('\nFeatures after preprocessing:')
print(df_clean.columns.tolist())

## 4. Train / Test Split

In [ ]:
X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

# Stratified split preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Training set : {X_train.shape[0]:,} samples  |  Fraud: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'Test set     : {X_test.shape[0]:,} samples  |  Fraud: {y_test.sum():,} ({y_test.mean()*100:.3f}%)')

## 5. Model Definitions

We use `class_weight='balanced'` where supported — this is a simple built-in technique
to handle imbalanced classes **without** resampling. It serves as our traditional baseline.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,     # needed for ROC-AUC
        random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE
    )
}

print(f'{len(models)} models defined and ready to train.')

## 6. Training & Evaluation

In [ ]:
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train, y_train)
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model'      : model,
        'y_pred'     : y_pred,
        'y_pred_prob': y_pred_prob,
        'precision'  : precision_score(y_test, y_pred),
        'recall'     : recall_score(y_test, y_pred),
        'f1'         : f1_score(y_test, y_pred),
        'roc_auc'    : roc_auc_score(y_test, y_pred_prob),
        'avg_prec'   : average_precision_score(y_test, y_pred_prob),
    }
    print(f'Done. F1={results[name]["f1"]:.4f}  ROC-AUC={results[name]["roc_auc"]:.4f}')

print('\nAll models trained.')

## 7. Results Summary Table

In [ ]:
summary = pd.DataFrame({
    name: {
        'Precision'        : round(v['precision'], 4),
        'Recall'           : round(v['recall'], 4),
        'F1 Score'         : round(v['f1'], 4),
        'ROC-AUC'          : round(v['roc_auc'], 4),
        'Avg Precision'    : round(v['avg_prec'], 4),
    }
    for name, v in results.items()
}).T

summary = summary.sort_values('F1 Score', ascending=False)

print('=== Model Performance Summary (Test Set) ===')
display(summary.style.highlight_max(color='lightgreen').highlight_min(color='lightsalmon'))

## 8. Classification Reports

In [ ]:
for name, v in results.items():
    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(classification_report(y_test, v['y_pred'], target_names=['Genuine', 'Fraud']))

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (name, v) in enumerate(results.items()):
    cm = confusion_matrix(y_test, v['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Genuine', 'Fraud'],
        yticklabels=['Genuine', 'Fraud'],
        ax=axes[idx]
    )
    tn, fp, fn, tp = cm.ravel()
    axes[idx].set_title(f'{name}\nTP={tp}  FP={fp}  FN={fn}  TN={tn}', fontsize=11)
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrices — Traditional Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

## 10. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for (name, v), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, v['y_pred_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC = {v['roc_auc']:.4f})", color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Traditional Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

## 11. Precision-Recall Curves

> **Why PR curves matter here:** With heavy class imbalance, Precision-Recall curves are often more informative than ROC curves, as they focus on the minority class (fraud).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for (name, v), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, v['y_pred_prob'])
    ax.plot(rec, prec, label=f"{name} (AP = {v['avg_prec']:.4f})", color=color, lw=2)

# Baseline = random classifier = fraud rate
baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', lw=1.5, label=f'Random Classifier (baseline={baseline:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Traditional Models', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pr_curves.png', bbox_inches='tight')
plt.show()

## 12. Feature Importances (Tree-based Models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

tree_models = {
    'Random Forest'     : results['Random Forest']['model'],
    'Gradient Boosting' : results['Gradient Boosting']['model'],
}

for ax, (name, model) in zip(axes, tree_models.items()):
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    top15 = importances.nlargest(15).sort_values()
    
    top15.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name} — Top 15 Features', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Feature Importances', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importances.png', bbox_inches='tight')
plt.show()

## 13. Metric Comparison Bar Charts

In [ ]:
metrics = ['Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(metrics))
bar_width = 0.2

fig, ax = plt.subplots(figsize=(13, 6))

for i, (name, v) in enumerate(results.items()):
    values = [v['precision'], v['recall'], v['f1'], v['roc_auc']]
    ax.bar(x + i * bar_width, values, bar_width, label=name, color=colors[i], edgecolor='white')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison — Traditional Classifiers', fontsize=14, fontweight='bold')
ax.set_xticks(x + bar_width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

## 14. Cross-Validation (Stratified K-Fold)

In [ ]:
# Cross-validation on full dataset for robustness check
# Using roc_auc as scoring metric

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:22s}  ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}')

# Boxplot
fig, ax = plt.subplots(figsize=(10, 5))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(ax=ax, patch_artist=True)
ax.set_title('5-Fold Cross-Validation ROC-AUC Scores', fontsize=13, fontweight='bold')
ax.set_ylabel('ROC-AUC')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cross_validation.png', bbox_inches='tight')
plt.show()

## 15. Best Model Selection & Export

In [ ]:
# Select best model by F1 score
best_name = max(results, key=lambda k: results[k]['f1'])
best = results[best_name]

print(f'Best traditional model: {best_name}')
print(f'  Precision  : {best["precision"]:.4f}')
print(f'  Recall     : {best["recall"]:.4f}')
print(f'  F1 Score   : {best["f1"]:.4f}')
print(f'  ROC-AUC    : {best["roc_auc"]:.4f}')

# Optionally save the model
import pickle

with open('best_traditional_model.pkl', 'wb') as f:
    pickle.dump(best['model'], f)

print(f'\nModel saved to best_traditional_model.pkl')

## 16. Observations & Next Steps

### Key Observations

| Concern | Finding |
|---|---|
| **Class Imbalance** | Fraud makes up ~0.17% of all transactions — heavily imbalanced |
| **Metric choice** | Accuracy is misleading; F1 and ROC-AUC are the relevant metrics |
| **class_weight='balanced'** | A simple, effective baseline technique for imbalanced data |
| **False Negatives** | Missing fraud is costly — recall is especially important |

### Next Steps → Data Augmentation Notebook
The `a_fool_fraud_data_augmentation.ipynb` notebook will:
1. Apply **SMOTE** and **ADASYN** to oversample the minority class in the training set
2. Explore **GAN-based** synthetic data generation
3. Re-train the same classifiers on augmented data
4. Compare all metrics against this traditional baseline